# Filtered Trades Analysis
Model + analysis container trades collected with current filters:
- `yes_price ∈ [0.40, 0.97)`, `time_remaining ∈ [100, 285]`, `edge ∈ [0.01, 0.15]`, YES only, `pct_change_open ≠ 0`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pyarrow.parquet as pq
from pathlib import Path

DATA = Path('../data/trades')
FEE = 0.02

model = pq.read_table(DATA / 'model_trades.parquet').to_pandas()
analysis = pq.read_table(DATA / 'analysis_trades.parquet').to_pandas()

# drop the handful of unresolved trades
model = model[model['pnl'].notna()].copy()
analysis = analysis[analysis['pnl'].notna()].copy()

print(f'Model:    {len(model):,} trades  |  {model["market_id"].nunique()} markets  |  {model["opened_at"].min().date()} → {model["opened_at"].max().date()}')
print(f'Analysis: {len(analysis):,} trades  |  {analysis["market_id"].nunique()} markets  |  {analysis["opened_at"].min().date()} → {analysis["opened_at"].max().date()}')

## 1. Top-level summary

In [ ]:
def summary(df, name):
    wins = (df['pnl'] > 0).sum()
    total_pnl = df['pnl'].sum()
    roi = 100 * total_pnl / len(df)
    avg_edge = df['edge'].mean()
    avg_yes = df['yes_price'].mean()
    commission = FEE / df['yes_price']  # fee paid per winning trade (as fraction of stake)
    est_commission = (df['pnl'] > 0).sum() * commission[df['pnl'] > 0].mean()
    print(f'── {name} ──')
    print(f'  Trades:      {len(df):,}')
    print(f'  Win rate:    {100*wins/len(df):.1f}%  ({wins:,} wins)')
    print(f'  Total P&L:   ${total_pnl:+.2f}')
    print(f'  ROI/trade:   {roi:+.2f}%')
    print(f'  Avg edge:    {avg_edge:.4f}')
    print(f'  Avg yes_px:  {avg_yes:.3f}')
    print(f'  Est. fees:   ${est_commission:.2f}')
    print()

summary(model, 'Model (logistic regression)')
summary(analysis, 'Analysis (empirical lookup)')

## 2. Per-candle P&L  *(true independent unit)*

In [ ]:
def per_candle(df):
    """Collapse all trades on the same market into one row."""
    g = df.groupby('market_id').agg(
        pnl=('pnl', 'sum'),
        trades=('pnl', 'count'),
        resolved_yes=('resolved_yes', 'first'),
        avg_edge=('edge', 'mean'),
        avg_yes=('yes_price', 'mean'),
        avg_time=('time_remaining', 'mean'),
        opened_at=('opened_at', 'min'),
    ).reset_index()
    return g

mc = per_candle(model)
ac = per_candle(analysis)

def candle_summary(c, name):
    wins = (c['pnl'] > 0).sum()
    print(f'── {name} — per candle ──')
    print(f'  Candles:     {len(c):,}')
    print(f'  Win rate:    {100*wins/len(c):.1f}%')
    print(f'  Total P&L:   ${c["pnl"].sum():+.2f}')
    print(f'  Median P&L:  ${c["pnl"].median():+.2f}')
    print(f'  Max drawdown candle: ${c["pnl"].min():+.2f}')
    print(f'  Best candle: ${c["pnl"].max():+.2f}')
    print(f'  Avg trades/candle: {c["trades"].mean():.1f}')
    print()

candle_summary(mc, 'Model')
candle_summary(ac, 'Analysis')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, c, name, color in zip(axes, [mc, ac], ['Model', 'Analysis'], ['steelblue', 'darkorange']):
    c_sorted = c.sort_values('opened_at')
    cumulative = c_sorted['pnl'].cumsum()
    ax.plot(range(len(cumulative)), cumulative.values, color=color, linewidth=1.5)
    ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
    ax.set_title(f'{name} — cumulative P&L (per candle)')
    ax.set_xlabel('Candle index')
    ax.set_ylabel('P&L ($)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, c, name, color in zip(axes, [mc, ac], ['Model', 'Analysis'], ['steelblue', 'darkorange']):
    ax.hist(c['pnl'], bins=40, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    ax.axvline(c['pnl'].mean(), color='red', linewidth=1.5, linestyle='-', label=f'mean={c["pnl"].mean():+.2f}')
    ax.set_title(f'{name} — per-candle P&L distribution')
    ax.set_xlabel('P&L ($)')
    ax.set_ylabel('Candles')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. P&L by edge bucket

In [ ]:
def edge_breakdown(df, name, color, ax_pnl, ax_win):
    bins = np.arange(0.01, 0.16, 0.01)
    df = df.copy()
    df['edge_bucket'] = pd.cut(df['edge'], bins=bins)
    g = df.groupby('edge_bucket', observed=True).agg(
        total_pnl=('pnl', 'sum'),
        win_rate=('pnl', lambda x: (x > 0).mean()),
        n=('pnl', 'count'),
    )
    labels = [f'{b.left:.2f}' for b in g.index]
    x = range(len(g))

    bars = ax_pnl.bar(x, g['total_pnl'], color=[color if v >= 0 else 'crimson' for v in g['total_pnl']], alpha=0.8)
    for i, (v, n) in enumerate(zip(g['total_pnl'], g['n'])):
        ax_pnl.text(i, v + (0.3 if v >= 0 else -0.6), f'n={n}', ha='center', fontsize=6.5)
    ax_pnl.set_xticks(x)
    ax_pnl.set_xticklabels(labels, rotation=45, fontsize=8)
    ax_pnl.axhline(0, color='black', linewidth=0.7)
    ax_pnl.set_title(f'{name} — P&L by edge')
    ax_pnl.set_ylabel('Total P&L ($)')
    ax_pnl.grid(True, alpha=0.3, axis='y')

    ax_win.bar(x, g['win_rate'] * 100, color=color, alpha=0.6)
    ax_win.axhline(50, color='black', linewidth=0.7, linestyle='--')
    ax_win.set_xticks(x)
    ax_win.set_xticklabels(labels, rotation=45, fontsize=8)
    ax_win.set_title(f'{name} — win rate by edge')
    ax_win.set_ylabel('Win rate (%)')
    ax_win.set_ylim(0, 100)
    ax_win.grid(True, alpha=0.3, axis='y')

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
edge_breakdown(model, 'Model', 'steelblue', axes[0][0], axes[1][0])
edge_breakdown(analysis, 'Analysis', 'darkorange', axes[0][1], axes[1][1])
plt.tight_layout()
plt.show()

## 4. P&L by time remaining (30s bins)

In [ ]:
def time_breakdown(df, name, color, ax):
    df = df.copy()
    bins = list(range(100, 300, 30))
    df['time_bucket'] = pd.cut(df['time_remaining'], bins=bins)
    g = df.groupby('time_bucket', observed=True).agg(
        total_pnl=('pnl', 'sum'),
        win_rate=('pnl', lambda x: 100 * (x > 0).mean()),
        n=('pnl', 'count'),
    )
    labels = [f'{int(b.left)}–{int(b.right)}s' for b in g.index]
    x = range(len(g))
    bars = ax.bar(x, g['total_pnl'], color=[color if v >= 0 else 'crimson' for v in g['total_pnl']], alpha=0.8)
    for i, (v, n, wr) in enumerate(zip(g['total_pnl'], g['n'], g['win_rate'])):
        ax.text(i, v + (0.5 if v >= 0 else -1.5), f'n={n}\n{wr:.0f}%', ha='center', fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, fontsize=8)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_title(f'{name} — P&L by time remaining')
    ax.set_ylabel('Total P&L ($)')
    ax.grid(True, alpha=0.3, axis='y')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
time_breakdown(model, 'Model', 'steelblue', axes[0])
time_breakdown(analysis, 'Analysis', 'darkorange', axes[1])
plt.tight_layout()
plt.show()

## 5. P&L by yes_price bucket (0.05 bins)

In [ ]:
def price_breakdown(df, name, color, ax):
    df = df.copy()
    bins = np.arange(0.40, 1.00, 0.05)
    df['price_bucket'] = pd.cut(df['yes_price'], bins=bins)
    g = df.groupby('price_bucket', observed=True).agg(
        total_pnl=('pnl', 'sum'),
        win_rate=('pnl', lambda x: 100 * (x > 0).mean()),
        n=('pnl', 'count'),
    )
    labels = [f'{b.left:.2f}' for b in g.index]
    x = range(len(g))
    ax.bar(x, g['total_pnl'], color=[color if v >= 0 else 'crimson' for v in g['total_pnl']], alpha=0.8)
    for i, (v, n, wr) in enumerate(zip(g['total_pnl'], g['n'], g['win_rate'])):
        ax.text(i, v + (0.5 if v >= 0 else -1.5), f'n={n}\n{wr:.0f}%', ha='center', fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, fontsize=8)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_title(f'{name} — P&L by yes_price')
    ax.set_ylabel('Total P&L ($)')
    ax.grid(True, alpha=0.3, axis='y')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
price_breakdown(model, 'Model', 'steelblue', axes[0])
price_breakdown(analysis, 'Analysis', 'darkorange', axes[1])
plt.tight_layout()
plt.show()

## 6. P&L by BTC % change bucket

In [ ]:
def pct_breakdown(df, name, color, ax):
    df = df.copy()
    bins = np.arange(-0.01, 0.012, 0.002)
    df['pct_bucket'] = pd.cut(df['pct_change_open'], bins=bins)
    g = df.groupby('pct_bucket', observed=True).agg(
        total_pnl=('pnl', 'sum'),
        win_rate=('pnl', lambda x: 100 * (x > 0).mean()),
        n=('pnl', 'count'),
    )
    labels = [f'{b.mid*100:+.1f}%' for b in g.index]
    x = range(len(g))
    ax.bar(x, g['total_pnl'], color=[color if v >= 0 else 'crimson' for v in g['total_pnl']], alpha=0.8)
    for i, (v, n) in enumerate(zip(g['total_pnl'], g['n'])):
        ax.text(i, v + (0.3 if v >= 0 else -0.8), f'n={n}', ha='center', fontsize=6.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, fontsize=7)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_title(f'{name} — P&L by BTC % change')
    ax.set_ylabel('Total P&L ($)')
    ax.grid(True, alpha=0.3, axis='y')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
pct_breakdown(model, 'Model', 'steelblue', axes[0])
pct_breakdown(analysis, 'Analysis', 'darkorange', axes[1])
plt.tight_layout()
plt.show()

## 7. Model calibration: predicted_prob vs actual win rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, name, color in zip(axes, [model, analysis], ['Model', 'Analysis'], ['steelblue', 'darkorange']):
    df = df.copy()
    bins = np.arange(0.40, 1.00, 0.05)
    df['pred_bucket'] = pd.cut(df['predicted_prob'], bins=bins)
    g = df.groupby('pred_bucket', observed=True).agg(
        actual_win=('resolved_yes', lambda x: (x == True).mean()),
        n=('resolved_yes', 'count'),
        pred_mid=('predicted_prob', 'mean'),
    ).dropna()

    ax.scatter(g['pred_mid'], g['actual_win'], s=g['n'] / g['n'].max() * 300, color=color, alpha=0.7, zorder=3)
    ax.plot([0.4, 1.0], [0.4, 1.0], 'k--', linewidth=1, label='perfect calibration')
    for _, row in g.iterrows():
        ax.annotate(f'n={int(row["n"])}', (row['pred_mid'], row['actual_win']),
                    textcoords='offset points', xytext=(4, 4), fontsize=7)
    ax.set_xlim(0.35, 1.0)
    ax.set_ylim(0.35, 1.0)
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Actual win rate')
    ax.set_title(f'{name} — calibration')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Trades per candle vs P&L — is concentration a factor?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, c, name, color in zip(axes, [mc, ac], ['Model', 'Analysis'], ['steelblue', 'darkorange']):
    ax.scatter(c['trades'], c['pnl'], alpha=0.35, s=20, color=color)
    ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
    # trend line
    z = np.polyfit(c['trades'], c['pnl'], 1)
    p = np.poly1d(z)
    xs = np.linspace(c['trades'].min(), c['trades'].max(), 100)
    ax.plot(xs, p(xs), color='red', linewidth=1.5, label=f'slope={z[0]:+.3f}')
    ax.set_xlabel('Trades in candle')
    ax.set_ylabel('Candle P&L ($)')
    ax.set_title(f'{name} — P&L vs trades/candle')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Edge × time heatmap (mean P&L per trade)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

edge_bins = np.arange(0.01, 0.16, 0.02)
time_bins = list(range(100, 300, 30))

for ax, df, name in zip(axes, [model, analysis], ['Model', 'Analysis']):
    df = df.copy()
    df['eb'] = pd.cut(df['edge'], bins=edge_bins, labels=[f'{b:.2f}' for b in edge_bins[:-1]])
    df['tb'] = pd.cut(df['time_remaining'], bins=time_bins, labels=[f'{b}s' for b in time_bins[:-1]])
    pivot = df.groupby(['tb', 'eb'], observed=True)['pnl'].mean().unstack('eb')
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_xlabel('Edge bucket')
    ax.set_ylabel('Time remaining')
    ax.set_title(f'{name} — mean P&L per trade (edge × time)')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## 10. Model vs Analysis — same candle comparison

In [ ]:
shared = mc.merge(ac, on='market_id', suffixes=('_model', '_analysis'))
print(f'Candles in both: {len(shared)}')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(shared['pnl_analysis'], shared['pnl_model'], alpha=0.3, s=15, color='purple')
lim = max(abs(shared['pnl_model']).max(), abs(shared['pnl_analysis']).max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=1, label='equal P&L')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
corr = shared['pnl_model'].corr(shared['pnl_analysis'])
ax.set_xlabel('Analysis P&L ($)')
ax.set_ylabel('Model P&L ($)')
ax.set_title(f'Model vs Analysis per candle  (corr={corr:.3f})')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

both_win = ((shared['pnl_model'] > 0) & (shared['pnl_analysis'] > 0)).sum()
both_lose = ((shared['pnl_model'] < 0) & (shared['pnl_analysis'] < 0)).sum()
print(f'Both profitable:  {both_win}/{len(shared)}  ({100*both_win/len(shared):.1f}%)')
print(f'Both losing:      {both_lose}/{len(shared)}  ({100*both_lose/len(shared):.1f}%)')
print(f'Model better:     {(shared["pnl_model"] > shared["pnl_analysis"]).sum()}')
print(f'Analysis better:  {(shared["pnl_analysis"] > shared["pnl_model"]).sum()}')

## 11. Interactive filter explorer

In [ ]:
import ipywidgets as widgets
from IPython.display import display

source_w    = widgets.ToggleButtons(options=['Model', 'Analysis'], description='Source:')
edge_min_w  = widgets.FloatSlider(value=0.01, min=0.01, max=0.14, step=0.01, description='Edge min:')
edge_max_w  = widgets.FloatSlider(value=0.15, min=0.02, max=0.15, step=0.01, description='Edge max:')
time_min_w  = widgets.IntSlider(value=100, min=100, max=280, step=15, description='Time min:')
time_max_w  = widgets.IntSlider(value=285, min=115, max=285, step=15, description='Time max:')
price_min_w = widgets.FloatSlider(value=0.40, min=0.40, max=0.90, step=0.05, description='Price min:')
price_max_w = widgets.FloatSlider(value=0.97, min=0.45, max=0.97, step=0.05, description='Price max:')

out = widgets.Output()

def update(*_):
    df = model.copy() if source_w.value == 'Model' else analysis.copy()
    mask = (
        (df['edge'] >= edge_min_w.value) & (df['edge'] <= edge_max_w.value) &
        (df['time_remaining'] >= time_min_w.value) & (df['time_remaining'] <= time_max_w.value) &
        (df['yes_price'] >= price_min_w.value) & (df['yes_price'] <= price_max_w.value)
    )
    filtered = df[mask]
    with out:
        out.clear_output(wait=True)
        if len(filtered) == 0:
            print('No trades match filters.')
            return
        wins = (filtered['pnl'] > 0).sum()
        total_pnl = filtered['pnl'].sum()
        roi = 100 * total_pnl / len(filtered)
        c = per_candle(filtered)
        print(f'Trades: {len(filtered):,}  |  Candles: {len(c)}  |  Win rate: {100*wins/len(filtered):.1f}%')
        print(f'Total P&L: ${total_pnl:+.2f}  |  ROI/trade: {roi:+.2f}%  |  P&L/candle: ${c["pnl"].mean():+.2f}')

        fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
        c_sorted = c.sort_values('opened_at')
        axes[0].plot(c_sorted['pnl'].cumsum().values, linewidth=1.5)
        axes[0].axhline(0, color='black', linewidth=0.7, linestyle='--')
        axes[0].set_title('Cumulative P&L (per candle)')
        axes[0].set_xlabel('Candle index')
        axes[0].grid(True, alpha=0.3)

        edge_bins = np.arange(edge_min_w.value, edge_max_w.value + 0.01, 0.01)
        filtered['eb'] = pd.cut(filtered['edge'], bins=edge_bins)
        g = filtered.groupby('eb', observed=True)['pnl'].sum()
        axes[1].bar(range(len(g)), g.values,
                    color=['steelblue' if v >= 0 else 'crimson' for v in g.values], alpha=0.8)
        axes[1].set_xticks(range(len(g)))
        axes[1].set_xticklabels([f'{b.left:.2f}' for b in g.index], rotation=45, fontsize=8)
        axes[1].axhline(0, color='black', linewidth=0.7)
        axes[1].set_title('P&L by edge bucket')
        axes[1].grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.show()

for w in [source_w, edge_min_w, edge_max_w, time_min_w, time_max_w, price_min_w, price_max_w]:
    w.observe(update, names='value')

display(widgets.VBox([
    source_w,
    widgets.HBox([edge_min_w, edge_max_w]),
    widgets.HBox([time_min_w, time_max_w]),
    widgets.HBox([price_min_w, price_max_w]),
    out
]))
update()